# Data preparation for FLUX Light-Curve example (GRB)

In [3]:
import os
import subprocess
from pathlib import Path
from cosipy.util import fetch_wasabi_file
from cosipy import BinnedData
from cosipy.pipeline.task.task import cosi_bindata
#
indir=Path.cwd() # Current path by default
#

Download all files

In [ ]:
#
#Get Response from the develop folder in wasabi (new version)
#
filename = "ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"
fetch_wasabi_file('COSI-SMEX/develop/Data/Responses/' + filename, 
                  output=indir/filename,
                  checksum = '7121f094be50e7bfe9b31e53015b0e85')

#
#Get Orientation files
#
filename="DC3_final_530km_3_month_with_slew_1sbins_GalacticEarth_SAA.ori"
fetch_wasabi_file('COSI-SMEX/DC3/Data/Orientation/'+filename, 
                  output=indir/filename,
                  checksum = 'b87fd41b6c28a5c0c51448ce2964e57c')

#
#Get Galactic background
#
filename='GalTotal_SA100_F98_3months_unbinned_data_filtered_with_SAAcut.fits.gz'
fetch_wasabi_file('COSI-SMEX/DC3/Data/Backgrounds/Ge/'+filename, 
                  output=indir/filename,
                  checksum = '9fda5a7b15a90358abc2b886979f9fef')

#
#Get GRB source data
#GRB
#
filename="GRB_bn081207680_3months_unbinned_data_filtered_with_SAAcut.fits.gz"
fetch_wasabi_file('COSI-SMEX/DC3/Data/Sources/'+filename, 
                  output=indir/filename,
                  checksum = '570a0ef083044123a811fb95d803098a')

In [ ]:
#Bin GRB data using the app cosi-bindata
#
args=['--config','prep_lctutorial_data.yaml', '--overwrite','--config_group','bindata_grb','--suffix','grbdc3']
cosi_bindata (argv=args)
#
#Bin and cut in time the bk data
#
args=['--config','prep_lctutorial_data.yaml', '--overwrite','--config_group','bindata_bk','--suffix','galbk']
cosi_bindata (argv=args)
#
#==================================
#
#Combine grb and galactic background, to have a dataset for the fit
#
grb=BinnedData("bin_grbdc3.yaml")
#
grb_bk=os.path.join (indir,"galbk_grbdc3")
#
grb.combine_unbinned_data(["GRB_bn081207680_3months_unbinned_data_filtered_with_SAAcut.fits.gz","GalTotal_SA100_F98_3months_unbinned_data_filtered_with_SAAcut.fits.gz"], output_name=grb_bk)
#
#Bin the grb+bk file:
#
args=['--config','prep_lctutorial_data.yaml', '--overwrite','--config_group','bindata_grbbk','--suffix','galbk_grbdc3']
cosi_bindata(argv=args)